In [7]:
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf

from multiprocessing import Pool

from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import Input, Conv1D, Flatten, Dense, Permute
from tensorflow.keras.regularizers import l2

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from numpy.lib.stride_tricks import as_strided

## System Calls List

In [2]:
syscalls = [
"sys_enter_llistxattr",
"sys_enter_setgroups",
"sys_enter_lremovexattr",
"sys_enter_sethostname",
"sys_enter_accept",
"sys_enter_lseek",
"sys_enter_setitimer",
"sys_enter_accept4",
"sys_enter_lsetxattr",
"sys_enter_setns",
"sys_enter_acct",
"sys_enter_madvise",
"sys_enter_setpgid",
"sys_enter_add_key",
"sys_enter_mbind",
"sys_enter_setpriority",
"sys_enter_adjtimex",
"sys_enter_membarrier",
"sys_enter_setregid",
"sys_enter_personality",
"sys_enter_memfd_create",
"sys_enter_setresgid",
"sys_enter_bind",
"sys_enter_memfd_secret",
"sys_enter_setresuid",
"sys_enter_bpf",
"sys_enter_migrate_pages",
"sys_enter_setreuid",
"sys_enter_brk",
"sys_enter_mincore",
"sys_enter_setrlimit",
"sys_enter_capget",
"sys_enter_mkdirat",
"sys_enter_setsid",
"sys_enter_capset",
"sys_enter_mknodat",
"sys_enter_setsockopt",
"sys_enter_chdir",
"sys_enter_mlock",
"sys_enter_settimeofday",
"sys_enter_chroot",
"sys_enter_mlock2",
"sys_enter_setuid",
"sys_enter_clock_adjtime",
"sys_enter_mlockall",
"sys_enter_setxattr",
"sys_enter_clock_getres",
"sys_enter_mmap",
"sys_enter_shmat",
"sys_enter_clock_gettime",
"sys_enter_mount",
"sys_enter_shmctl",
"sys_enter_clock_nanosleep",
"sys_enter_mount_setattr",
"sys_enter_shmdt",
"sys_enter_clock_settime",
"sys_enter_move_mount",
"sys_enter_shmget",
"sys_enter_clone",
"sys_enter_move_pages",
"sys_enter_shutdown",
"sys_enter_clone3",
"sys_enter_mprotect",
"sys_enter_sigaltstack",
"sys_enter_close",
"sys_enter_mq_getsetattr",
"sys_enter_signalfd4",
"sys_enter_close_range",
"sys_enter_mq_notify",
"sys_enter_socket",
"sys_enter_connect",
"sys_enter_mq_open",
"sys_enter_socketpair",
"sys_enter_copy_file_range",
"sys_enter_mq_timedreceive",
"sys_enter_splice",
"sys_enter_delete_module",
"sys_enter_mq_timedsend",
"sys_enter_statfs",
"sys_enter_dup",
"sys_enter_mq_unlink",
"sys_enter_statx",
"sys_enter_dup3",
"sys_enter_mremap",
"sys_enter_swapoff",
"sys_enter_epoll_create1",
"sys_enter_msgctl",
"sys_enter_swapon",
"sys_enter_epoll_ctl",
"sys_enter_msgget",
"sys_enter_symlinkat",
"sys_enter_epoll_pwait",
"sys_enter_msgrcv",
"sys_enter_sync",
"sys_enter_epoll_pwait2",
"sys_enter_msgsnd",
"sys_enter_sync_file_range",
"sys_enter_eventfd2",
"sys_enter_msync",
"sys_enter_syncfs",
"sys_enter_execve",
"sys_enter_munlock",
"sys_enter_sysinfo",
"sys_enter_execveat",
"sys_enter_munlockall",
"sys_enter_syslog",
"sys_enter_exit",
"sys_enter_munmap",
"sys_enter_tee",
"sys_enter_exit_group",
"sys_enter_name_to_handle_at",
"sys_enter_tgkill",
"sys_enter_faccessat",
"sys_enter_nanosleep",
"sys_enter_timer_create",
"sys_enter_faccessat2",
"sys_enter_newfstat",
"sys_enter_timer_delete",
"sys_enter_fadvise64",
"sys_enter_newfstatat",
"sys_enter_timer_getoverrun",
"sys_enter_fallocate",
"sys_enter_newuname",
"sys_enter_timer_gettime",
"sys_enter_fanotify_init",
"sys_enter_open_by_handle_at",
"sys_enter_timer_settime",
"sys_enter_fanotify_mark",
"sys_enter_open_tree",
"sys_enter_timerfd_create",
"sys_enter_fchdir",
"sys_enter_openat",
"sys_enter_timerfd_gettime",
"sys_enter_fchmod",
"sys_enter_openat2",
"sys_enter_timerfd_settime",
"sys_enter_fchmodat",
"sys_enter_perf_event_open",
"sys_enter_times",
"sys_enter_fchown",
"sys_enter_pidfd_getfd",
"sys_enter_tkill",
"sys_enter_fchownat",
"sys_enter_pidfd_open",
"sys_enter_truncate",
"sys_enter_fcntl",
"sys_enter_pidfd_send_signal",
"sys_enter_umask",
"sys_enter_fdatasync",
"sys_enter_pipe2",
"sys_enter_umount",
"sys_enter_fgetxattr",
"sys_enter_pivot_root",
"sys_enter_unlinkat",
"sys_enter_finit_module",
"sys_enter_ppoll",
"sys_enter_unshare",
"sys_enter_flistxattr",
"sys_enter_prctl",
"sys_enter_userfaultfd",
"sys_enter_flock",
"sys_enter_pread64",
"sys_enter_utimensat",
"sys_enter_fremovexattr",
"sys_enter_preadv",
"sys_enter_vhangup",
"sys_enter_fsconfig",
"sys_enter_preadv2",
"sys_enter_vmsplice",
"sys_enter_fsetxattr",
"sys_enter_prlimit64",
"sys_enter_wait4",
"sys_enter_fsmount",
"sys_enter_process_madvise",
"sys_enter_waitid",
"sys_enter_fsopen",
"sys_enter_process_mrelease",
"sys_enter_write",
"sys_enter_fspick",
"sys_enter_process_vm_readv",
"sys_enter_writev",
"sys_enter_fstatfs",
"sys_enter_process_vm_writev",
"sys_enter_fsync",
"sys_enter_pselect6",
"sys_enter_ftruncate",
"sys_enter_ptrace",
"sys_enter_futex",
"sys_enter_pwrite64",
"sys_enter_get_mempolicy",
"sys_enter_pwritev",
"sys_enter_get_robust_list",
"sys_enter_pwritev2",
"sys_enter_getcpu",
"sys_enter_quotactl",
"sys_enter_getcwd",
"sys_enter_quotactl_fd",
"sys_enter_getdents64",
"sys_enter_read",
"sys_enter_getegid",
"sys_enter_readahead",
"sys_enter_geteuid",
"sys_enter_readlinkat",
"sys_enter_getgid",
"sys_enter_readv",
"sys_enter_getgroups",
"sys_enter_reboot",
"sys_enter_getitimer",
"sys_enter_recvfrom",
"sys_enter_getpeername",
"sys_enter_recvmmsg",
"sys_enter_getpgid",
"sys_enter_recvmsg",
"sys_enter_getpid",
"sys_enter_remap_file_pages",
"sys_enter_getppid",
"sys_enter_removexattr",
"sys_enter_getpriority",
"sys_enter_renameat",
"sys_enter_getrandom",
"sys_enter_renameat2",
"sys_enter_getresgid",
"sys_enter_request_key",
"sys_enter_getresuid",
"sys_enter_restart_syscall",
"sys_enter_getrlimit",
"sys_enter_rseq",
"sys_enter_getrusage",
"sys_enter_rt_sigaction",
"sys_enter_getsid",
"sys_enter_rt_sigpending",
"sys_enter_getsockname",
"sys_enter_rt_sigprocmask",
"sys_enter_getsockopt",
"sys_enter_rt_sigqueueinfo",
"sys_enter_gettid",
"sys_enter_rt_sigreturn",
"sys_enter_gettimeofday",
"sys_enter_rt_sigsuspend",
"sys_enter_getuid",
"sys_enter_rt_sigtimedwait",
"sys_enter_getxattr",
"sys_enter_rt_tgsigqueueinfo",
"sys_enter_init_module",
"sys_enter_sched_get_priority_max",
"sys_enter_inotify_add_watch",
"sys_enter_sched_get_priority_min",
"sys_enter_inotify_init1",
"sys_enter_sched_getaffinity",
"sys_enter_inotify_rm_watch",
"sys_enter_sched_getattr",
"sys_enter_io_cancel",
"sys_enter_sched_getparam",
"sys_enter_io_destroy",
"sys_enter_sched_getscheduler",
"sys_enter_io_getevents",
"sys_enter_sched_rr_get_interval",
"sys_enter_io_pgetevents",
"sys_enter_sched_setaffinity",
"sys_enter_io_setup",
"sys_enter_sched_setattr",
"sys_enter_io_submit",
"sys_enter_sched_setparam",
"sys_enter_io_uring_enter",
"sys_enter_sched_setscheduler",
"sys_enter_io_uring_register",
"sys_enter_sched_yield",
"sys_enter_io_uring_setup",
"sys_enter_seccomp",
"sys_enter_ioctl",
"sys_enter_semctl",
"sys_enter_ioprio_get",
"sys_enter_semget",
"sys_enter_ioprio_set",
"sys_enter_semop",
"sys_enter_kcmp",
"sys_enter_semtimedop",
"sys_enter_kexec_file_load",
"sys_enter_sendfile64",
"sys_enter_kexec_load",
"sys_enter_sendmmsg",
"sys_enter_keyctl",
"sys_enter_sendmsg",
"sys_enter_kill",
"sys_enter_sendto",
"sys_enter_landlock_add_rule",
"sys_enter_set_mempolicy",
"sys_enter_landlock_create_ruleset",
"sys_enter_set_robust_list",
"sys_enter_landlock_restrict_self",
"sys_enter_set_tid_address",
"sys_enter_lgetxattr",
"sys_enter_setdomainname",
"sys_enter_linkat",
"sys_enter_setfsgid",
"sys_enter_listen",
"sys_enter_setfsuid",
"sys_enter_listxattr",
"sys_enter_setgid"
]

## Loading CSV

In [8]:
NUM_CLASSES = 4
CLASSES = np.array(['benign', 'sysrv', 'xmrig', 'mirai'])
DATASET_DIR = "../dataset/phase2"
VECTOR_LENGTH = 32 * 32
DT = 30

label_encoder = LabelEncoder()
label_encoder.fit(syscalls)

def csvToVector(file_path):
    try:
        data = pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        data = pd.read_csv(file_path, encoding='ISO-8859-1')

    data_encoded = label_encoder.fit_transform(data['SYSTEM_CALL'])
    vector = np.zeros(VECTOR_LENGTH, dtype=np.float32)
    syscall_nums = min(len(data_encoded), VECTOR_LENGTH)
    vector[:syscall_nums] = data_encoded[:syscall_nums]

    return vector / 299.0

def process_file(args):
    file_path, class_idx = args
    vector = csvToVector(file_path)
    return vector, class_idx

def load_data(dataset_dir):
    x = []
    y = []
    classes = [f"0/{DT}sec_0", f"1/{DT}sec_1", f"2/{DT}sec_2", f"3/{DT}sec_3"]

    file_paths = []
    for class_idx, class_name in enumerate(classes):
        class_dir = os.path.join(dataset_dir, class_name)
        for file_name in os.listdir(class_dir):
            if file_name.endswith('.csv'):
                file_path = os.path.join(class_dir, file_name)
                file_paths.append((file_path, class_idx))

    with Pool() as pool:
        results = pool.map(process_file, file_paths)

    x, y = zip(*results)
    x = np.array(x)
    y = np.array(y)

    return x, y

In [9]:
X, y = load_data(DATASET_DIR)

## Train, Validation, Test Split and Normalize

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

X_train = X_train.reshape(-1, VECTOR_LENGTH, 1)
X_val = X_val.reshape(-1, VECTOR_LENGTH, 1)
X_test = X_test.reshape(-1, VECTOR_LENGTH, 1)

classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}

## 1D CNN model

In [11]:
input_layer = Input(shape=(VECTOR_LENGTH, 1))

x = Conv1D(filters=16, kernel_size=3, strides=10, padding='valid', activation='relu')(input_layer)
x = Conv1D(filters=32, kernel_size=3, strides=1, padding='valid', activation='relu')(x)
x = Conv1D(filters=64, kernel_size=3, strides=1, padding='valid', activation='relu')(x)

x = Permute((2, 1))(x)
x = Flatten()(x)

x = Dense(32, activation='relu', kernel_regularizer=l2(1e-4))(x)
output_layer = Dense(NUM_CLASSES, activation='softmax', kernel_regularizer=l2(1e-4))(x)

model = Model(input_layer, output_layer)

opt = Adam(learning_rate=0.001)
model.compile(loss='sparse_categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

## Check Point

In [12]:
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=0.00001)
checkpoint = ModelCheckpoint(
    filepath='./phase2_30sec.h5',
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

## Model Training

In [13]:
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_val, y_val), class_weight=class_weight, callbacks=[reduce_lr, checkpoint])

Epoch 1/100
91/94 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7630 - loss: 0.8278
Epoch 1: val_accuracy improved from None to 0.90400, saving model to ./phase2_30sec.h5



Epoch 1: finished saving model to ./phase2_30sec.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.7678 - loss: 0.8161 - val_accuracy: 0.9040 - val_loss: 0.4835 - learning_rate: 0.0010
Epoch 2/100
89/94 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9062 - loss: 0.3755
Epoch 2: val_accuracy improved from 0.90400 to 0.90933, saving model to ./phase2_30sec.h5



Epoch 2: finished saving model to ./phase2_30sec.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9069 - loss: 0.3713 - val_accuracy: 0.9093 - val_loss: 0.3284 - learning_rate: 0.0010
Epoch 3/100
91/94 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9111 - loss: 0.3131
Epoch 3: val_accuracy improved from 0.90933 to 0.91867, saving model to ./phase2_30sec.h5



Epoch 3: finished saving model to ./phase2_30sec.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9112 - loss: 0.3128 - val_accuracy: 0.9187 - val_loss: 0.2720 - learning_rate: 0.0010
Epoch 4/100
89/94 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9094 - loss: 0.3187
Epoch 4: val_accuracy improved from 0.91867 to 0.92533, saving model to ./phase2_30sec.h5



Epoch 4: finished saving model to ./phase2_30sec.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9109 - loss: 0.3142 - val_accuracy: 0.9253 - val_loss: 0.2529 - learning_rate: 0.0010
Epoch 5/100
91/94 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9227 - loss: 0.2762
Epoch 5: val_accuracy improved from 0.92533 to 0.92800, saving model to ./phase2_30sec.h5


...


Epoch 19: finished saving model to ./phase2_30sec.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9543 - loss: 0.1464 - val_accuracy: 0.9533 - val_loss: 0.1417 - learning_rate: 0.0010
Epoch 20/100
92/94 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9565 - loss: 0.1475
Epoch 20: val_accuracy did not improve from 0.95333
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9570 - loss: 0.1468 - val_accuracy: 0.9533 - val_loss: 0.1553 - learning_rate: 0.0010
Epoch 21/100
93/94 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9620 - loss: 0.1306
Epoch 21: val_accuracy did not improve from 0.95333
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9623 - loss: 0.1304 - val_accuracy: 0.9507 - val_loss: 0.1716 - learning_rate: 0.0010
Epoch 22/100
91/94 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9639 - loss: 0.1222
Epoch 22: val_accuracy improved from 0.95333 to 0.96800, saving model to ./phase2_30sec.h5



Epoch 22: finished saving model to ./phase2_30sec.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9640 - loss: 0.1225 - val_accuracy: 0.9680 - val_loss: 0.1335 - learning_rate: 0.0010
Epoch 23/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9700 - loss: 0.1073
Epoch 23: val_accuracy did not improve from 0.96800
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9700 - loss: 0.1073 - val_accuracy: 0.9667 - val_loss: 0.1395 - learning_rate: 0.0010
Epoch 24/100
90/94 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9747 - loss: 0.0998
Epoch 24: val_accuracy did not improve from 0.96800
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9750 - loss: 0.1003 - val_accuracy: 0.9680 - val_loss: 0.1320 - learning_rate: 0.0010
Epoch 25/100
93/94 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9704 - loss: 0.1035
Epoch 25: val_accuracy did not improve from 0.96800
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9703 - loss: 0.1040 - val_accuracy: 0.9547 - val_loss: 


Epoch 27: finished saving model to ./phase2_30sec.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.9733 - loss: 0.1002 - val_accuracy: 0.9693 - val_loss: 0.1224 - learning_rate: 0.0010
Epoch 28/100
92/94 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9759 - loss: 0.0942
Epoch 28: val_accuracy did not improve from 0.96933
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9750 - loss: 0.0950 - val_accuracy: 0.9680 - val_loss: 0.1332 - learning_rate: 0.0010
Epoch 29/100
93/94 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9812 - loss: 0.0813
Epoch 29: val_accuracy did not improve from 0.96933
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9810 - loss: 0.0815 - val_accuracy: 0.9653 - val_loss: 0.1403 - learning_rate: 0.0010
Epoch 30/100
89/94 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9793 - loss: 0.0890
Epoch 30: val_accuracy did not improve from 0.96933
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9783 - loss: 0.0921 - val_accuracy: 0.9693 - val_loss: 


Epoch 32: finished saving model to ./phase2_30sec.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.9756 - loss: 0.0895 - val_accuracy: 0.9733 - val_loss: 0.1296 - learning_rate: 0.0010
Epoch 33/100
91/94 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9811 - loss: 0.0729
Epoch 33: val_accuracy did not improve from 0.97333
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9813 - loss: 0.0732 - val_accuracy: 0.9733 - val_loss: 0.1255 - learning_rate: 0.0010
Epoch 34/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9847 - loss: 0.0752
Epoch 34: val_accuracy did not improve from 0.97333
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.9847 - loss: 0.0752 - val_accuracy: 0.9707 - val_loss: 0.1401 - learning_rate: 0.0010
Epoch 35/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9823 - loss: 0.0799
Epoch 35: val_accuracy did not improve from 0.97333
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9823 - loss: 0.0799 - val_accuracy: 0.9733 - val_loss: 


Epoch 38: finished saving model to ./phase2_30sec.h5
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9873 - loss: 0.0606 - val_accuracy: 0.9773 - val_loss: 0.1425 - learning_rate: 5.0000e-04
Epoch 39/100
91/94 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9890 - loss: 0.0577
Epoch 39: val_accuracy did not improve from 0.97733
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9887 - loss: 0.0579 - val_accuracy: 0.9680 - val_loss: 0.1328 - learning_rate: 5.0000e-04
Epoch 40/100
91/94 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9894 - loss: 0.0591
Epoch 40: val_accuracy did not improve from 0.97733
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9893 - loss: 0.0586 - val_accuracy: 0.9680 - val_loss: 0.1434 - learning_rate: 5.0000e-04
Epoch 41/100
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9863 - loss: 0.0585
Epoch 41: val_accuracy did not improve from 0.97733
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.9863 - loss: 0.0585 - val_accuracy: 0.9707 

## Evaluate (float32)

In [14]:
cp_model = load_model('./phase2_30sec.h5')
cp_model.evaluate(X_test, y_test, batch_size=1000)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.9682 - loss: 0.2037 


[0.20367279648780823, 0.9682440757751465]

In [15]:
y_pred = cp_model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes, target_names=list(CLASSES), digits=4))

51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
              precision    recall  f1-score   support

      benign     0.9969    0.9658    0.9811       673
       sysrv     0.9102    0.9545    0.9319       308
       xmrig     0.9806    0.9712    0.9759       313
       mirai     0.9564    0.9840    0.9700       312

    accuracy                         0.9682      1606
   macro avg     0.9610    0.9689    0.9647      1606
weighted avg     0.9693    0.9682    0.9685      1606



In [16]:
conf_matrix = confusion_matrix(y_test, y_pred_classes)
conf_matrix_df = pd.DataFrame(conf_matrix, index=list(CLASSES), columns=list(CLASSES))
print("Confusion Matrix:")
print(conf_matrix_df)

Confusion Matrix:
        benign  sysrv  xmrig  mirai
benign     650     17      3      3
sysrv        2    294      3      9
xmrig        0      7    304      2
mirai        0      5      0    307


## Q15 Quantization

In [17]:
def scale_for_q15(w):
    absmax = np.abs(w).max()
    if absmax <= 1.0:
        return w, 1
    scale = 1
    while absmax / scale > 1.0:
        scale *= 2
    return w / scale, scale

def to_q15(x):
    return np.clip(np.round(x * 32768.0), -32768, 32767).astype(np.int64)

def to_q15_bias(x):
    return np.clip(np.round(x * 32768.0), -2**31, 2**31 - 1).astype(np.int64)

def quantize_weights_flatten(model):
    conv_layers = [l for l in model.layers if 'conv1d' in l.name]
    dense_layers = [l for l in model.layers if l.name.startswith('dense')]
    c1_w_f, c1_b_f = conv_layers[0].get_weights()
    c2_w_f, c2_b_f = conv_layers[1].get_weights()
    c3_w_f, c3_b_f = conv_layers[2].get_weights()
    d_w_f, d_b_f = dense_layers[0].get_weights()
    fc_w_f, fc_b_f = dense_layers[1].get_weights()
    c1_w = np.transpose(c1_w_f[:, 0, :], (1, 0))
    c2_w = np.transpose(c2_w_f, (2, 1, 0))
    c3_w = np.transpose(c3_w_f, (2, 1, 0))
    d_w = d_w_f.T
    fc_w = fc_w_f.T
    out = {}
    for name, w, b in [('conv1', c1_w, c1_b_f), ('conv2', c2_w, c2_b_f), ('conv3', c3_w, c3_b_f),
                        ('dense', d_w, d_b_f), ('fc2', fc_w, fc_b_f)]:
        w_s, scale = scale_for_q15(w)
        out[f'{name}_w'] = to_q15(w_s)
        out[f'{name}_b'] = to_q15_bias(b / scale)
    return out

def windows_1d(a, out_len, k, stride, axis):
    a = np.ascontiguousarray(a)
    shape = list(a.shape); shape[axis] = out_len; shape = shape + [k]
    strides = list(a.strides); ts = strides[axis]
    strides[axis] = ts * stride; strides = strides + [ts]
    return as_strided(a, shape=shape, strides=strides)

def q15_forward(qw, X, s1, s2, s3, seq_len):
    c1w, c1b = qw['conv1_w'].astype(np.int64), qw['conv1_b'].astype(np.int64)
    c2w, c2b = qw['conv2_w'].astype(np.int64), qw['conv2_b'].astype(np.int64)
    c3w, c3b = qw['conv3_w'].astype(np.int64), qw['conv3_b'].astype(np.int64)
    dw, db = qw['dense_w'].astype(np.int64), qw['dense_b'].astype(np.int64)
    fcw, fcb = qw['fc2_w'].astype(np.int64), qw['fc2_b'].astype(np.int64)
    F1, K1 = c1w.shape
    F2, _, K2 = c2w.shape
    F3, _, K3 = c3w.shape
    c1out = (seq_len - K1) // s1 + 1
    c2out = (c1out - K2) // s2 + 1
    c3out = (c2out - K3) // s3 + 1
    Xq = np.clip(np.round(X * 32768), -32768, 32767).astype(np.int64)
    win1 = windows_1d(Xq, c1out, K1, s1, axis=1)
    a1 = np.maximum(0, (np.einsum('ntk,fk->nft', win1, c1w) >> 15) + c1b[None, :, None])
    a1c = np.clip(a1, -32768, 32767)
    win2 = windows_1d(a1c, c2out, K2, s2, axis=2)
    a2 = np.maximum(0, (np.einsum('nctk,fck->nft', win2, c2w) >> 15) + c2b[None, :, None])
    a2c = np.clip(a2, -32768, 32767)
    win3 = windows_1d(a2c, c3out, K3, s3, axis=2)
    a3 = np.maximum(0, (np.einsum('nctk,fck->nft', win3, c3w) >> 15) + c3b[None, :, None])
    a3c = np.clip(a3, -32768, 32767)
    N = X.shape[0]
    flat = a3c.reshape(N, F3 * c3out)
    hidden = np.clip(np.maximum(0, (flat @ dw.T >> 15) + db[None, :]), -32768, 32767)
    logits = (hidden @ fcw.T >> 15) + fcb[None, :]
    return np.argmax(logits, axis=1)

## Evaluate (Q15)

In [18]:
qw = quantize_weights_flatten(cp_model)
X_test_flat = X_test.reshape(-1, VECTOR_LENGTH)
y_pred_q15 = q15_forward(qw, X_test_flat, 10, 1, 1, VECTOR_LENGTH)

print(classification_report(y_test, y_pred_q15, target_names=list(CLASSES), digits=4))

              precision    recall  f1-score   support

      benign     0.9767    0.9346    0.9552       673
       sysrv     0.7923    0.5325    0.6369       308
       xmrig     0.0667    0.0064    0.0117       313
       mirai     0.4069    0.9455    0.5689       312

    accuracy                         0.6787      1606
   macro avg     0.5606    0.6047    0.5432      1606
weighted avg     0.6533    0.6787    0.6352      1606



In [19]:
conf_matrix_q15 = confusion_matrix(y_test, y_pred_q15)
conf_matrix_q15_df = pd.DataFrame(conf_matrix_q15, index=list(CLASSES), columns=list(CLASSES))
print("Confusion Matrix (Q15):")
print(conf_matrix_q15_df)

Confusion Matrix (Q15):
        benign  sysrv  xmrig  mirai
benign     629     21      1     22
sysrv       15    164     27    102
xmrig        0      5      2    306
mirai        0     17      0    295
